In [0]:
%sql
MERGE INTO catalog_ventas.gold.fact_ventas tgt
USING (
SELECT *  FROM (
SELECT
    v.venta,
    df.id_fecha,
    dp.id_articulo,
    dcli.id_cliente,
    dc.id_usulogin,
    dmp.id_medio_pago,
    v.delivery,
    v.vtaestado,
    v.precio,
    v.cant,
    v.total,
    ROW_NUMBER() OVER (
        PARTITION BY v.venta
        ORDER BY v._processing_timestamp DESC
    ) AS rn

FROM catalog_ventas.silver.ventas_clean v
LEFT JOIN catalog_ventas.gold.dim_fecha df ON v.vtafecha = df.vtafecha
LEFT JOIN catalog_ventas.gold.dim_producto dp ON v.articulo = dp.articulo
LEFT JOIN catalog_ventas.gold.dim_clientes dcli ON v.cliente = dcli.cliente
LEFT JOIN catalog_ventas.gold.dim_cajero dc ON v.usulogin = dc.usulogin
LEFT JOIN catalog_ventas.gold.dim_medio_pago dmp ON v.medio_pago = dmp.medio_pago

)s
WHERE rn = 1

) src

ON tgt.venta = src.venta 

WHEN MATCHED AND 
(
    COALESCE(tgt.total,0) <> COALESCE(src.total,0)
    OR COALESCE(tgt.cant,0) <> COALESCE(src.cant,0)
    OR COALESCE(tgt.vtaestado,'') <> COALESCE(src.vtaestado,'')
) 
THEN UPDATE SET
    tgt.id_fecha = src.id_fecha,
    tgt.id_cliente = src.id_cliente,
    tgt.id_usulogin = src.id_usulogin,
    tgt.id_medio_pago = src.id_medio_pago,
    tgt.delivery = src.delivery,
    tgt.vtaestado = src.vtaestado,
    tgt.precio = src.precio,
    tgt.cant = src.cant,
    tgt.total = src.total,
    tgt._refresh_timestamp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
INSERT (
    venta,
    id_fecha,
    id_articulo,
    id_cliente,
    id_usulogin,
    id_medio_pago,
    delivery,
    vtaestado,
    precio,
    cant,
    total,
    _refresh_timestamp
)
VALUES (

    src.venta,
    src.id_fecha,
    src.id_articulo,
    src.id_cliente,
    src.id_usulogin,
    src.id_medio_pago,
    src.delivery,
    src.vtaestado,
    src.precio,
    src.cant,
    src.total,
    CURRENT_TIMESTAMP()
)

In [0]:
%sql
/*
OPTIMIZE catalog_ventas.gold.fact_ventas
ZORDER BY (id_articulo, id_cliente,id_medio_pago);
*/